# AutoData Agent Colab Demo

A beginner-friendly notebook for running the AutoData Agent smoke or prototype pipeline on Google Colab.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
!pip install -q pyyaml datasets transformers accelerate peft trl bitsandbytes tqdm

## 3. Clone GitHub Repo Or Load Local Project Files

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/autodata-agent.git'
PROJECT_DIR = '/content/autodata-agent'
USE_LOCAL_FILES = os.path.exists(PROJECT_DIR)

if not USE_LOCAL_FILES:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

## 4. Set Run Mode

In [ ]:
RUN_MODE = 'smoke'
USE_REAL_MODEL = False
USE_REAL_TRAINING = False
USE_MOCK_GENERATION = True

# Later, try:
# RUN_MODE = 'prototype'
# USE_REAL_MODEL = True
# USE_REAL_TRAINING = True
# USE_MOCK_GENERATION = False

OUTPUT_DIR = '/content/drive/MyDrive/AutoData/outputs'

## 5. Load YAML Config

In [ ]:
from pathlib import Path
import yaml

config_path = Path('configs') / f'{RUN_MODE}_colab.yaml'
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['project']['output_dir'] = OUTPUT_DIR
config['models']['use_real_model'] = USE_REAL_MODEL
config['training']['enabled'] = USE_REAL_TRAINING
config['training']['dry_run'] = not USE_REAL_TRAINING
config['generation']['use_mock_generation'] = USE_MOCK_GENERATION
config['generation']['provider'] = 'mock' if USE_MOCK_GENERATION else config['generation'].get('provider', 'local_hf')
config['dataset']['use_mock_data'] = not USE_REAL_MODEL

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(yaml.safe_dump(config, sort_keys=False))

## 6. Prepare MedMCQA Data

In [ ]:
from autodata.config import save_config
from autodata.data.medmcqa_loader import load_medmcqa_data
from autodata.utils.io import create_timestamped_run_dir
from autodata.utils.seed import set_seed

set_seed(config['project']['seed'])
run_dir = create_timestamped_run_dir(config['project']['output_dir'])
save_config(config, run_dir / 'config.yaml')
eval_examples, train_pool = load_medmcqa_data(config, run_dir=run_dir)
print('Run directory:', run_dir)
print('Eval examples:', len(eval_examples))
print('Train pool examples:', len(train_pool))

## 7. Run Base Model Evaluation

In [ ]:
from autodata.evaluation.evaluator import Evaluator
from autodata.utils.gpu import clear_gpu_memory
from autodata.utils.io import write_json

evaluator = Evaluator(config)
evaluation_base = evaluator.evaluate(eval_examples, phase='base')
write_json(run_dir / 'evaluation_base.json', evaluation_base)
clear_gpu_memory()
print('Base accuracy:', evaluation_base.overall_accuracy)
print('Weakest domain:', evaluation_base.weakest_domain)

## 8. Diagnose Weak Domains

In [ ]:
from autodata.diagnosis.weakness_diagnoser import WeaknessDiagnoser

diagnosis = WeaknessDiagnoser(config).diagnose(evaluation_base)
write_json(run_dir / 'diagnosis.json', diagnosis)
print(diagnosis.summary)

## 9. Create Data Generation Plan

In [ ]:
from autodata.planning.data_planner import DataPlanner

data_plan = DataPlanner(config).create_plan(evaluation_base, diagnosis)
write_json(run_dir / 'data_plan.json', data_plan)
for domain, item in data_plan.plan.items():
    print(domain, item.num_samples, '-', item.reason)

## 10. Generate Synthetic SFT Samples

In [ ]:
from autodata.generation.generator import DataGenerator
from autodata.utils.io import write_jsonl

generated_samples = DataGenerator(config).generate(data_plan, round_id='round_1')
write_jsonl(run_dir / 'generated_samples.jsonl', generated_samples)
clear_gpu_memory()
print('Generated:', len(generated_samples))

## 11. Verify Generated Samples

In [ ]:
from autodata.verification.verifier import DataVerifier

verification = DataVerifier(config).verify(generated_samples, heldout_eval_examples=eval_examples)
write_jsonl(run_dir / 'verified_samples.jsonl', verification.accepted)
write_jsonl(run_dir / 'rejected_samples.jsonl', verification.rejected)
write_json(run_dir / 'verification_report.json', verification.report)
print(verification.report)

## 12. Build Training Mixture

In [ ]:
from autodata.mixture.mixture_optimizer import MixtureOptimizer

mixture = MixtureOptimizer(config).optimize(verification.accepted, data_plan, evaluation_base)
mixture_report = {
    'final_sample_count': len(mixture.samples),
    'domain_distribution': mixture.domain_distribution,
    'strategy': mixture.strategy,
    'dropped_samples': mixture.dropped_samples,
    'reason_for_mixture_decisions': mixture.reasons,
}
write_jsonl(run_dir / 'mixture_train.jsonl', mixture.samples)
write_json(run_dir / 'mixture_report.json', mixture_report)
print(mixture_report)

## 13. Run LoRA / QLoRA Fine-Tuning

In [ ]:
from autodata.training.trainer import Trainer

training_report = Trainer(config).train(mixture.samples, run_dir=run_dir)
write_json(run_dir / 'training_report.json', training_report)
clear_gpu_memory()
print(training_report)

## 14. Re-Evaluate Model

In [ ]:
model_path = training_report.output_dir if training_report.status == 'completed' else None
evaluation_after = evaluator.evaluate(eval_examples, model_path=model_path, phase='after')
write_json(run_dir / 'evaluation_after.json', evaluation_after)
clear_gpu_memory()
print('After accuracy:', evaluation_after.overall_accuracy)

## 15. Save Reports To Google Drive

In [ ]:
from autodata.evaluation.metrics import compare_evaluations, system_metrics

model_metrics = compare_evaluations(evaluation_base, evaluation_after)
auto_metrics = system_metrics(
    generated_samples=generated_samples,
    accepted_samples=verification.accepted,
    rejected_samples=verification.rejected,
    mixture_samples=mixture.samples,
    evaluation_gain=model_metrics['overall_gain'],
)
round_summary = {
    'run_dir': str(run_dir),
    'base_overall_accuracy': evaluation_base.overall_accuracy,
    'after_overall_accuracy': evaluation_after.overall_accuracy,
    'overall_gain': model_metrics['overall_gain'],
    'generated_count': len(generated_samples),
    'accepted_count': len(verification.accepted),
    'mixture_sample_count': len(mixture.samples),
    'training_status': training_report.status,
    'model_level': model_metrics,
    'system_level': auto_metrics,
}
write_json(run_dir / 'round_summary.json', round_summary)
print('Saved reports to:', run_dir)

## 16. Show Summary Tables

In [ ]:
import pandas as pd

domain_rows = []
for domain, before in evaluation_base.per_domain.items():
    after = evaluation_after.per_domain[domain]
    domain_rows.append({
        'domain': domain,
        'base_accuracy': before.accuracy,
        'after_accuracy': after.accuracy,
        'gain': after.accuracy - before.accuracy,
        'generated': auto_metrics['generated_data_count_per_domain'].get(domain, 0),
        'accepted': auto_metrics['accepted_data_count_per_domain'].get(domain, 0),
        'mixture': auto_metrics['mixture_distribution'].get(domain, 0),
    })

display(pd.DataFrame(domain_rows))
display(pd.DataFrame([round_summary]))